In [80]:
import os 
import json 
import requests 
from typing import List, Dict, Optional
from datetime import datetime 
import re 
import time 
from functools import wraps 
from langchain_core.tools import tool 
from bs4 import BeautifulSoup 
from openai import OpenAI

In [81]:
# =============================================================================
# LangChain v1.0.x - Latest Official Release
# WebSurfX + VLLM Single Agent (Modern API)
# =============================================================================

# Cell 1: v1.0.x IMPORTS
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate
from langchain_classic import hub
import requests
from bs4 import BeautifulSoup
import re
import time

print("✅ LangChain v1.0.x imports")

# Cell 2: TOOLS (v1.0.x Optimized)

WEBSURFX_URL = "http://0.0.0.0:8080"
VLLM_URL = "http://127.0.0.1:5000/v1"
VLLM_API_KEY = "no-key"
VLLM_MODEL_NAME = "cpatonn/Qwen3-30B-A3B-Thinking-2507-AWQ-4bit"
@tool
def search_web(query: str) -> str:
    """WebSurf-X search tool"""
    try:
        # WebSurf-X returns HTML even with ?format=json
        # Extract results from HTML structure
        resp = requests.get("http://localhost:8080/search", 
                           params={"q": query}, timeout=12)
        
        soup = BeautifulSoup(resp.text, "html.parser")
        
        # WebSurf-X result selectors (from your HTML)
        results = soup.select(".result") or soup.select("article") or soup.select(".entry")
        
        if not results:
            # Fallback: all links
            results = soup.select("a[href*='http']")[:10]
        
        output = f"WebSurf-X Results for '{query}':\n\n"
        for i, result in enumerate(results[:8], 1):
            title_elem = result.select_one("h3, .title, h2, h1") or result
            link_elem = result.select_one("a")
            
            title = title_elem.get_text(strip=True)[:80]
            url = link_elem.get("href") if link_elem else ""
            
            if url and title and "http" in url:
                snippet = result.get_text(strip=True)[:120]
                output += f"{i}. **{title}**\n"
                output += f"   {url}\n"
                output += f"   {snippet}...\n\n"
        
        return output if len(output) > 100 else "No results found"
        
    except Exception as e:
        return f"[WebSurf-X error: {str(e)}]"

@tool
def get_content(urls: list[str]) -> str:
    """Extract content from URLs"""
    content = ""
    for url in urls[:3]:
        try:
            r = requests.get(url, timeout=10)
            soup = BeautifulSoup(r.content, "html.parser")
            
            # Clean content
            for tag in soup(["script", "style", "nav", "footer"]):
                tag.decompose()
            
            text = soup.get_text(separator=" ", strip=True)[:2500]
            title = soup.title.string if soup.title else url.split("/")[-1]
            
            content += f"\n## {title}\n\n{text[:600]}...\n\n"
            time.sleep(0.5)
        except Exception:
            continue
    
    return content or "No content retrieved"

@tool
def summarize_content(text: str, query: str) -> str:
    """Summarize content using VLLM"""
    from openai import OpenAI
    
    client = OpenAI(base_url=VLLM_URL, api_key=VLLM_API_KEY)
    
    prompt = f"""Original query: "{query}"

Summarize the most relevant information from this content:

{text[:4000]}

Provide a concise, accurate summary:"""
    
    try:
        response = client.chat.completions.create(
            model=VLLM_MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=500
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Summarization failed: {str(e)}"

tools = [search_web, get_content, summarize_content]
print(f"✅ {len(tools)} v1.0.x tools loaded")






✅ LangChain v1.0.x imports
✅ 3 v1.0.x tools loaded


In [82]:
# Cell 3: v1.0.x AGENT (Modern ReAct)
llm = ChatOpenAI(
    model=VLLM_MODEL_NAME,
    base_url=VLLM_URL,
    api_key=VLLM_API_KEY,
    temperature=0.7
)

# Pull official ReAct prompt (v1.0.x standard)
react_prompt = hub.pull("hwchase17/react")

# Create modern ReAct agent
agent = create_react_agent(llm, tools, react_prompt)

# v1.0.x Executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

print("✅ v1.0.x ReAct agent ready!")
print("Usage: agent_executor.invoke({\"input\": \"query\"})")


✅ v1.0.x ReAct agent ready!
Usage: agent_executor.invoke({"input": "query"})


In [83]:
# Cell 4: CUSTOM v1.0.x PROMPT (Optional)
research_template = """Answer the question step by step using the tools.

Question: {input}

TOOLS ({tool_names}):

Use this exact format:
Thought: [your reasoning]
Action: [tool name]  
Action Input: [tool input]
Observation: [tool output]
... repeat ...
Thought: I have enough info
Final Answer: [your answer]

Begin!

{agent_scratchpad}"""

custom_prompt = PromptTemplate.from_template(research_template)

print("✅ Custom prompt ready")


✅ Custom prompt ready


In [84]:
# Cell 5: LIVE TEST
test_query = "President of USA"
print(f"\n🧪 v1.0.x Agent Test: {test_query}")
print("=" * 60)

result = agent_executor.invoke({"input": test_query})
print("\n✅ RESULT:")
print(result["output"])



🧪 v1.0.x Agent Test: President of USA


> Entering new AgentExecutor chain...


Okay, so the user is asking for the President of the USA. Let me think about how to approach this.

First, I need to confirm who the current president is. I know that Joe Biden is the current president, but I should verify using the tools provided to make sure I'm up-to-date. The tools available are search_web, get_content, and summarize_content.

Since the question is straightforward, maybe a quick web search would give the latest info. Let me use search_web with the query "current president of USA". That should fetch the most recent data.

Wait, but the user might be asking for the president as of the time the tool is being used. Let me check if there's any possibility of a recent election. The 2024 election is coming up, but if it's before November 2024, Biden is still the president. But since the current date in the context isn't specified, I'll assume it's the present time.

Using search_web("current president of USA") should return the correct info. Let me do that.

After the sea

In [85]:
# Cell 6: PRODUCTION READY FUNCTIONS
V10_AGENT = agent_executor

def research(query: str):
    """One-line research"""
    return V10_AGENT.invoke({"input": query})

def batch_research(queries: list[str]):
    """Batch research"""
    return [research(q) for q in queries]

print("\n🚀 Production functions:")
print("research('your query')")
print("batch_research(['q1', 'q2'])")




🚀 Production functions:
research('your query')
batch_research(['q1', 'q2'])


In [86]:
# Cell 7: FULL EXPORT
EXPORT = {
    "agent": V10_AGENT,
    "tools": tools,
    "llm": llm,
    "research": research,
    "batch_research": batch_research,
    "config": {"vllm": VLLM_URL, "websurfx": WEBSURFX_URL}
}

print("\n💾 Exported to EXPORT dict")
print("EXPORT['research']('query')")



💾 Exported to EXPORT dict
EXPORT['research']('query')


In [90]:
query="what is the capital of France?"

In [91]:
agent_executor.invoke({"input": query})



> Entering new AgentExecutor chain...
Okay, the user is asking for the capital of France. Let me think. I remember that Paris is the capital, but I should confirm using the tools provided.

First, I should use the search_web tool to look up the capital of France. Let me input the query "capital of France" into search_web.

Wait, but maybe I should check if there's any chance of a trick question. No, France's capital is definitely Paris. But to be thorough, I'll use the tools as instructed.

So, Action: search_web with query "capital of France". The tool will return some search results.

Then, I need to get the content from the URLs provided by the search. Let's say the search results have a URL like example.com/capital-france. I'll use get_content on that URL.

After getting the content, I might need to summarize it to confirm the capital. But since it's a straightforward question, the summary should state Paris.

Wait, the tools are given, so I have to follow the process. Let's simu

{'input': 'what is the capital of France?',
 'output': 'The capital of France is Paris.'}